In [1]:
%load_ext autoreload
%autoreload 2

In [13]:
import os, sys, re, random, time
import importlib
import pandas as pd

import data_loaders

importlib.reload(data_loaders)

<module 'data_loaders' from 'C:\\Beryllium Base\\WORKSPACES\\MTECH\\RS-project\\src\\data_loaders\\__init__.py'>

In [3]:
os.getcwd()

'C:\\Beryllium Base\\WORKSPACES\\MTECH\\RS-project\\sandbox'

In [4]:
data_model = data_loaders.KuaiRecData(root='../data/KuaiRec2.0/data', specimen='small')

In [7]:
matrix = data_model.matrix
print(len(matrix))
matrix.head()

4676570


,user_id,video_id,play_duration,video_duration,time,date,timestamp,watch_ratio
0,14,148,4381,6067,2020-07-05 05:27:48.378,20200705.0,1.593898e+09,0.722103
1,14,183,11635,6100,2020-07-05 05:28:00.057,20200705.0,1.593898e+09,1.907377
2,14,3649,22422,10867,2020-07-05 05:29:09.479,20200705.0,1.593898e+09,2.063311
3,14,5262,4479,7908,2020-07-05 05:30:43.285,20200705.0,1.593898e+09,0.566388
4,14,8234,4602,11000,2020-07-05 05:35:43.459,20200705.0,1.593899e+09,0.418364


In [8]:
user_groups = matrix.groupby('user_id')

In [12]:
K = 5  # history length

trajectories = []

for user, group in matrix.sort_values('timestamp').groupby('user_id'):
    items = group['video_id'].tolist()
    rewards = group['watch_ratio'].tolist()

    for i in range(K, len(items)):
        state = items[i-K:i]
        action = items[i]
        reward = rewards[i]
        next_state = items[i-K+1:i+1]

        trajectories.append((state, action, reward, next_state))

In [15]:
all_items = matrix['video_id'].unique().tolist()

def sample_candidates(true_item, num_neg=50):
    negatives = random.sample(all_items, num_neg)
    return [true_item] + negatives